# 02 — Crime Cleaning

**Notebook responsibility:** apply the cleaning decisions established by `01_data_ingestion_and_quality`'s audit, via `src/preprocessing/crime_cleaning.py`. Notebook orchestrates and documents; logic lives in `src/`.

**Aim:** turn `data/raw/crime/crime_raw_2021_2025.parquet` into `data/interim/crime_cleaned.parquet`, with a full audit trail (raw → transform → removed/flagged → final). No blind-dropping — every removal is justified against the notebook-01 findings.

**Cleaning decisions carried over from notebook 01 (see its Section 6 summary):**
| Issue | Decision | Rationale |
|---|---|---|
| Numeric fields stored as strings | Coerce to numeric | Socrata JSON returns everything as strings |
| `community_area` outside 1-77 (113 rows, 0.009%) | Drop | Unrecoverable for the community_area × date modeling unit |
| Missing lat/long/x/y (1.45%) | Keep as null | `community_area` is the spatial key, not coordinates |
| Duplicate `case_number` (155), 0 full-row dupes | Keep, log only | Confirmed distinct records sharing one case, not copies |
| Exact full-row duplicates (0 found so far) | Drop if any appear | Byte-identical copies only |

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import DATA_RAW, DATA_INTERIM
from src.preprocessing.crime_cleaning import clean_crime_data

pd.set_option("display.max_columns", 50)

_Findings: (fill in after running)_

## 1. Load Raw Crime Data

**Aim:** load the untouched Parquet produced by notebook 01. This file is never modified in place.

In [ ]:
crime_raw = pd.read_parquet(DATA_RAW / "crime" / "crime_raw_2021_2025.parquet")
print(f"Shape: {crime_raw.shape}")

_Findings: (confirm shape matches notebook-01 audit — 1,210,172 x 19)_

## 2. Apply Cleaning

**Aim:** run `clean_crime_data()` and inspect the returned audit dict before trusting the output — every number here should be explainable by the table in the header.

In [ ]:
crime_cleaned, audit = clean_crime_data(crime_raw)

for k, v in audit.items():
    print(f"{k:32s}: {v:,}")

_Findings: (fill in after running — sanity-check each audit number against the header table:
- `full_row_duplicates_dropped` should be 0 (notebook 01 found none)
- `invalid_community_area_dropped` should be 113
- `duplicate_case_numbers_kept` should be 155
- `missing_coordinates_kept` should be ~17,544 (1.45% of the post-drop row count)
- `final_rows` = `raw_rows` − `full_row_duplicates_dropped` − `invalid_community_area_dropped`
any mismatch means either the raw file changed since notebook 01 or the cleaning logic has a bug — stop and investigate, don't proceed.)_

## 3. Post-Clean Verification

**Aim:** confirm the cleaned frame actually satisfies the invariants the rest of the pipeline will assume — correct dtypes, no residual invalid community areas, no full-row duplicates.

In [ ]:
print(crime_cleaned.dtypes)
print()
assert crime_cleaned["community_area"].between(1, 77).all(), "invalid community_area survived cleaning"
assert crime_cleaned.duplicated().sum() == 0, "full-row duplicates survived cleaning"
assert pd.api.types.is_datetime64_any_dtype(crime_cleaned["date"]), "date not parsed to datetime"
print("All invariants hold.")

_Findings: (confirm dtypes: date=datetime64, community_area=int, arrest/domestic=bool, coordinate fields=float64 with nulls; assertions pass)_

## 4. Save Cleaned Data

**Aim:** persist to `data/interim/crime_cleaned.parquet` — the input for `03_spatiotemporal_feature_engineering`.

In [ ]:
out_path = DATA_INTERIM / "crime_cleaned.parquet"
crime_cleaned.to_parquet(out_path, index=False)
print(f"Saved {len(crime_cleaned):,} rows to: {out_path}")

_Findings: (confirm file written, row count matches `final_rows` from the audit)_

## 5. Summary

**Rows removed:** _(fill in — full-row duplicates + invalid community areas)_
**Rows retained:** _(fill in — final_rows / raw_rows as %)_
**Known residual issues carried forward (not fixed here, by design):**
- ~1.45% of rows have null coordinates — spatial plots in `04_crime_eda` must handle this explicitly (e.g. `.dropna(subset=["latitude","longitude"])` before mapping, not before aggregation).
- 155 case numbers still map to multiple records — if `03`'s daily aggregation counts by row (not by case), this is already correct; flag if any future step assumes one row = one case.
- `iucr` (367 codes) not yet grouped into broad categories — that mapping is Phase 4's job, done explicitly and documented, not here.

**Next step:** proceed to `03_spatiotemporal_feature_engineering.ipynb`.